## DSPy Prompts to Rewrite Question from Rationale

### Setting up DSPy

In [ ]:
# Import libraries
import dspy
import json

import os
from dotenv import load_dotenv
load_dotenv()
open_ai_api_key = os.getenv("OPENAI_API_KEY")


c:\Users\Cassandra\miniconda3\envs\coalesce\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Set up the LM (https://dspy-docs.vercel.app/api/language_model_clients/OpenAI)
gpt3_turbo = dspy.OpenAI(model='gpt-3.5-turbo', max_tokens=500, api_key=open_ai_api_key)  
dspy.configure(lm=gpt3_turbo)

In [3]:
gpt3_turbo("which openai model are you? are you gpt-3.5-turbo?")

['I am an AI assistant powered by the GPT-3 model developed by OpenAI. I am not specifically the GPT-3.5-turbo model, but I am based on the GPT-3 architecture.']

In [4]:
test_inputs = [
    {
        "question": {
            "cell_type": "question",
            "response_format": "open",
            "description": "",
            "main_text": "How much do you trust your local government's decisions on public parks and how often do you visit these parks?",
            "response_categories": []
        },
        "input_rationale": "The question lacks specificity by combining two distinct concepts, trust in local government decisions on public parks, and frequency of park visits, which can lead to confusion and inaccurate responses."
    }
]

In [10]:
constants = {
    "question_json_format": """{
        "cell_type": "question",
        "response_format": "open" or "closed",
        "description": string,
        "main_text": string,
        "response_categories": empty list for open questions or list of JSONs with an "id" and "text" field for closed questions
    }""",
    "json_formatting_message": """Return three JSON objects in a list that is surrounded by square brackets. Do not number the items in the list."""
}

In [11]:
def parse_json_str(json_str):
    return json.loads(json_str)

def fill_in_constants(input_str):
    for key in constants:
        input_str = input_str.replace("{"+key+"}", constants[key])
    print(input_str)
    return input_str

### Creating Signature

In [17]:
sig_description = """Rewrite a question to address the issues identified in the input_rationale. Please return 3 re-written questions."""

input_descriptions = """{
    "question": "The question to re-write. The input will be a JSON with the following structure: {question_json_format}",
    "input_rationale": "The rationale with potential issues in a question."}"""

output_descriptions = """{"rewritten_questions": "The 3 re-written questions. The output should be a list of JSONs where each element has the following structure: {question_json_format}. {json_formatting_message}"}"""

# convert the descriptions to JSON
input_descriptions_json = parse_json_str(input_descriptions)
output_descriptions_json = parse_json_str(output_descriptions)

class RewriteQuestion(dspy.Signature):

    question = dspy.InputField(desc=fill_in_constants(input_descriptions_json["question"]))
    input_rationale = dspy.InputField(desc=input_descriptions_json["input_rationale"])
    rewritten_questions = dspy.OutputField(desc=fill_in_constants(output_descriptions_json["rewritten_questions"]))

# set the signature description
RewriteQuestion.__doc__ = sig_description

print(RewriteQuestion.__doc__)

The question to re-write. The input will be a JSON with the following structure: {
        "cell_type": "question",
        "response_format": "open" or "closed",
        "description": string,
        "main_text": string,
        "response_categories": empty list for open questions or list of JSONs with an "id" and "text" field for closed questions
    }
The 3 re-written questions. The output should be a list of JSONs where each element has the following structure: {
        "cell_type": "question",
        "response_format": "open" or "closed",
        "description": string,
        "main_text": string,
        "response_categories": empty list for open questions or list of JSONs with an "id" and "text" field for closed questions
    }. Return three JSON objects in a list that is surrounded by square brackets. Do not number the items in the list.
Rewrite a question to address the issues identified in the input_rationale. Please return 3 re-written questions.


In [18]:
# Create a module
class RewriteQuestionModule(dspy.Module):
    def __init__(self):

        super().__init__()
        
        self.rewritten_questions = dspy.ChainOfThought(RewriteQuestion)

    def forward(self, question, input_rationale, return_rationale=False, temp=0.7):

        output = self.rewritten_questions(question=question, 
                                    input_rationale=input_rationale,
                                    config=dict(temperature=temp))

        # return the output as a dictionary

        if return_rationale:
            return {"rewritten_questions": output.rewritten_questions, "rationale": output.rationale}
        else:
            return {"rewritten_questions": output.rewritten_questions}

In [19]:
# Test CleanRationaleModule

rewrite_question_module = RewriteQuestionModule()

for test_input in test_inputs:

    test_input_str = json.dumps(test_input["question"])

    print("Input:")
    print(test_input_str)
    print(test_input["input_rationale"])
    print("\n")

    output = rewrite_question_module(test_input_str, 
                                    test_input["input_rationale"], 
                                    return_rationale=True)
    print(output)
    print("\n")

Input:
{"cell_type": "question", "response_format": "open", "description": "", "main_text": "How much do you trust your local government's decisions on public parks and how often do you visit these parks?", "response_categories": []}
The question lacks specificity by combining two distinct concepts, trust in local government decisions on public parks, and frequency of park visits, which can lead to confusion and inaccurate responses.


{'rewritten_questions': '[\n    { "cell_type": "question", "response_format": "open", "description": "Measure trust in local government decisions on public parks", "main_text": "How much do you trust your local government\'s decisions on public parks?", "response_categories": [] },\n    { "cell_type": "question", "response_format": "open", "description": "Assess frequency of park visits", "main_text": "How often do you visit public parks?", "response_categories": [] },\n    { "cell_type": "question", "response_format": "open", "description": "Understand 

In [20]:
# Let's look at the model history
gpt3_turbo.inspect_history(n=1)





Rewrite a question to address the issues identified in the input_rationale. Please return 3 re-written questions.

---

Follow the following format.

Question: The question to re-write. The input will be a JSON with the following structure: { "cell_type": "question", "response_format": "open" or "closed", "description": string, "main_text": string, "response_categories": empty list for open questions or list of JSONs with an "id" and "text" field for closed questions }

Input Rationale: The rationale with potential issues in a question.

Reasoning: Let's think step by step in order to ${produce the rewritten_questions}. We ...

Rewritten Questions: The 3 re-written questions. The output should be a list of JSONs where each element has the following structure: { "cell_type": "question", "response_format": "open" or "closed", "description": string, "main_text": string, "response_categories": empty list for open questions or list of JSONs with an "id" and "text" field for closed quest